In [ ]:
import os, warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
from sklearn.linear_model import LinearRegression
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")

TASK = "triclass"   # "binary" | "triclass"
SDF = "data/ChemTastesDB/ChemTastesDB_multilabel_properties.sdf"
OUT_DIR = "./outputs"
PROPS = ["MolecularWeight","XLogP","TPSA","HBondDonorCount","HBondAcceptorCount","RotatableBondCount","HeavyAtomCount","Complexity"]
NORM = {"bitter":"bitterness","bitterness":"bitterness","sweet":"sweetness","sweetness":"sweetness",
        "tasteless":"tastelessness","tastelessness":"tastelessness","no taste":"tastelessness","notaste":"tastelessness","none":"tastelessness"}
LMAP = {"bitterness":0,"sweetness":1} if TASK=="binary" else {"bitterness":0,"sweetness":1,"tastelessness":2}

In [ ]:
rows=[]
for mol in Chem.SDMolSupplier(SDF, sanitize=False, removeHs=False):
    if mol is None: continue
    try: Chem.SanitizeMol(mol)
    except Exception:
        try: Chem.SanitizeMol(mol, sanitizeOps=Chem.SanitizeFlags.SANITIZE_KEKULIZE)
        except Exception: continue
    tp = "Class taste" if mol.HasProp("Class taste") else next((p for p in mol.GetPropNames() if p.strip().lower()=="class taste"), None)
    if tp is None: continue
    key = NORM.get(mol.GetProp(tp).strip().lower(), mol.GetProp(tp).strip().lower())
    if key not in LMAP or mol.GetNumConformers()==0: continue
    v=[]
    for p in PROPS:
        try: x=float(mol.GetProp(p)); v.append(x if not np.isnan(x) else np.nan)
        except Exception: v.append(np.nan)
    rows.append(v)
df=pd.DataFrame(rows,columns=PROPS)
cc=df.dropna().reset_index(drop=True)
print(f"All {len(df)} | complete cases (all 8 properties measured) {len(cc)}")

In [ ]:
pear=cc.corr("pearson"); spear=cc.corr("spearman")
print("=== Pearson ==="); display(pear.round(3))
print("=== Spearman ==="); display(spear.round(3))
pairs=[]
for i,a in enumerate(PROPS):
    for b in PROPS[i+1:]:
        if abs(pear.loc[a,b])>=0.7 or abs(spear.loc[a,b])>=0.7:
            pairs.append({"a":a,"b":b,"pearson":pear.loc[a,b],"spearman":spear.loc[a,b]})
pairs=pd.DataFrame(pairs)
if len(pairs): pairs=pairs.sort_values("pearson",key=abs,ascending=False)
print("=== Pairs with |r| >= 0.7 ==="); display(pairs.round(3))

In [ ]:
X=cc.values; vif=[]
for i in range(X.shape[1]):
    Xo=np.delete(X,i,axis=1); r2=LinearRegression().fit(Xo,X[:,i]).score(Xo,X[:,i])
    vif.append({"property":PROPS[i],"R2_from_others":r2,"VIF":1/max(1e-8,1-r2)})
vif=pd.DataFrame(vif).sort_values("VIF",ascending=False).reset_index(drop=True)
display(vif.round(3))
print(f"VIF>5: {(vif.VIF>5).sum()}/8 | VIF>10: {(vif.VIF>10).sum()}/8")

In [ ]:
names=list(pear.columns); dist=1-pear.abs().values; np.fill_diagonal(dist,0)
order=leaves_list(linkage(squareform(dist,checks=False),method="average")); on=[names[i] for i in order]
mat=pear.loc[on,on].values
fig,ax=plt.subplots(figsize=(7,6)); im=ax.imshow(mat,vmin=-1,vmax=1,cmap="RdBu_r")
ax.set_xticks(range(8)); ax.set_yticks(range(8)); ax.set_xticklabels(on,rotation=45,ha="right",fontsize=9); ax.set_yticklabels(on,fontsize=9)
for i in range(8):
    for j in range(8): ax.text(j,i,f"{mat[i,j]:.2f}",ha="center",va="center",color="white" if abs(mat[i,j])>0.5 else "black",fontsize=8)
fig.colorbar(im,ax=ax,shrink=0.8,label="Pearson r"); fig.tight_layout()
fig.savefig(f"{OUT_DIR}/property_correlation_heatmap_{TASK}.png",dpi=300); plt.show()
pear.to_csv(f"{OUT_DIR}/property_pearson_{TASK}.csv"); spear.to_csv(f"{OUT_DIR}/property_spearman_{TASK}.csv"); vif.to_csv(f"{OUT_DIR}/property_vif_{TASK}.csv",index=False)
print("Saved:",OUT_DIR)

In [ ]:
import matplotlib as mpl
from matplotlib.gridspec import GridSpec
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["font.family"] = "Arial"

SHORT = {"MolecularWeight":"MW","XLogP":"XLogP","TPSA":"TPSA","HBondDonorCount":"HBD",
         "HBondAcceptorCount":"HBA","RotatableBondCount":"RotB","HeavyAtomCount":"HeavyAtom","Complexity":"Complexity"}

names = list(pear.columns); dist = 1 - pear.abs().values; np.fill_diagonal(dist, 0)
order = leaves_list(linkage(squareform(dist, checks=False), method="average"))
on = [names[i] for i in order]; lab = [SHORT[n] for n in on]

fig = plt.figure(figsize=(15, 6.6))
gs = GridSpec(1, 3, width_ratios=[1, 1, 0.04], wspace=0.28)
axs = [fig.add_subplot(gs[0]), fig.add_subplot(gs[1])]; cax = fig.add_subplot(gs[2])

for ax, corr, ttl in zip(axs, [pear, spear], ["Pearson", "Spearman"]):
    mat = corr.loc[on, on].values
    im = ax.imshow(mat, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(8)); ax.set_yticks(range(8))
    ax.set_xticklabels(lab, rotation=45, ha="right", rotation_mode="anchor", fontsize=11)
    ax.set_yticklabels(lab, fontsize=11)
    for i in range(8):
        for j in range(8):
            ax.text(j, i, f"{mat[i, j]:.3f}", ha="center", va="center", fontsize=8.5,
                    color="white" if abs(mat[i, j]) > 0.5 else "black")
    ax.set_title(ttl, fontsize=13, pad=8)

fig.colorbar(im, cax=cax, label="Correlation coefficient")
fig.savefig(f"{OUT_DIR}/FigureS2_property_correlation.pdf", bbox_inches="tight")
plt.show()

In [ ]:
import os, numpy as np, pandas as pd
from rdkit import Chem, RDLogger
from sklearn.metrics import f1_score, accuracy_score
RDLogger.DisableLog("rdApp.*")

TASK = "triclass"   # "binary" | "triclass"
SDF = "data/ChemTastesDB/ChemTastesDB_multilabel_properties.sdf"
BASE_DIR = "analysis/taste_baseline_cv"
TD = f"{BASE_DIR}/{TASK}"
STEP1 = f"{TD}/schnet_seed42/schnet_oof_probabilities_{TASK}_seed42.csv"
RF = f"{TD}/morgan_rf_oof_probabilities_{TASK}_seed42.csv"
OUT = os.path.expanduser(f"~/Desktop/property_missing_subset_{TASK}.csv")
B = 5000
PROPS = ["MolecularWeight","XLogP","TPSA","HBondDonorCount","HBondAcceptorCount","RotatableBondCount","HeavyAtomCount","Complexity"]
NORM = {"bitter":"bitterness","bitterness":"bitterness","sweet":"sweetness","sweetness":"sweetness",
        "tasteless":"tastelessness","tastelessness":"tastelessness","no taste":"tastelessness","notaste":"tastelessness","none":"tastelessness"}
LMAP = {"bitterness":0,"sweetness":1} if TASK=="binary" else {"bitterness":0,"sweetness":1,"tastelessness":2}

In [ ]:
labels, nmiss = [], []
for mol in Chem.SDMolSupplier(SDF, sanitize=False, removeHs=False):
    if mol is None: continue
    try: Chem.SanitizeMol(mol)
    except Exception:
        try: Chem.SanitizeMol(mol, sanitizeOps=Chem.SanitizeFlags.SANITIZE_KEKULIZE)
        except Exception: continue
    tp = "Class taste" if mol.HasProp("Class taste") else next((p for p in mol.GetPropNames() if p.strip().lower()=="class taste"), None)
    if tp is None: continue
    key = NORM.get(mol.GetProp(tp).strip().lower(), mol.GetProp(tp).strip().lower())
    if key not in LMAP or mol.GetNumConformers()==0: continue
    labels.append(LMAP[key])
    nmiss.append(sum(1 for p in PROPS if (not mol.HasProp(p)) or mol.GetProp(p).strip().lower() in ("","nan")))
labels, nmiss = np.array(labels), np.array(nmiss)
print(len(labels), pd.Series(nmiss).value_counts().sort_index().to_dict())

In [ ]:
s1 = pd.read_csv(STEP1).sort_values("sample_idx"); rf = pd.read_csv(RF).sort_values("sample_idx")
y = s1.true_label.values
assert len(y)==len(labels) and (y==labels).all(), "sample_idx alignment mismatch: SDF order differs from the OOF predictions"
print(f"Alignment OK: N={len(y)}")
preds = {"SchNet 3D-only": s1.pred_3d_only.values, "SchNet 3D+Property": s1.pred_3d_property.values, "Morgan-RF": rf.pred_morgan_rf.values}
groups = {"A complete (0 missing)": nmiss==0, "B partly missing (1-7)": (nmiss>=1)&(nmiss<=7),
          "C all missing (8)": nmiss==8, "D measured properties (0-7 missing)": nmiss<=7, "ALL": np.ones(len(y), bool)}

In [ ]:
mf = lambda a,b: f1_score(a,b,average="macro",zero_division=0)
rng = np.random.default_rng(42); rows=[]
for gname, mask in groups.items():
    yy=y[mask]; n=len(yy)
    r={"group":gname,"N":n,"n_tasteless":int((yy==2).sum()) if TASK=="triclass" else 0}
    for m,p in preds.items():
        r[f"{m} acc"]=accuracy_score(yy,p[mask]); r[f"{m} mF1"]=mf(yy,p[mask])
    a,b_=preds["SchNet 3D-only"][mask],preds["SchNet 3D+Property"][mask]
    d=np.array([mf(yy[i],b_[i])-mf(yy[i],a[i]) for i in (rng.integers(0,n,n) for _ in range(B))])
    r["Δ(3D+Prop - 3D-only) mF1"]=mf(yy,b_)-mf(yy,a)
    r["Δ CI low"],r["Δ CI high"]=np.percentile(d,[2.5,97.5])
    r["Δ p"]=max(min(1,2*min((d<=0).mean(),(d>=0).mean())),1/B)
    rows.append(r)
out=pd.DataFrame(rows)
pd.set_option("display.width",250); pd.set_option("display.max_columns",30)
display(out.round(4).T)
out.to_csv(OUT,index=False); print("Saved:",OUT)